In [12]:
import boto3
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo


In [13]:

# Create an AWS session using my configured profile
session = boto3.Session(
    profile_name="nordic-energy",
    region_name="eu-north-1"
)

# Create an S3 client to communicate with AWS
s3 = session.client("s3")

# Define the bucket where we will store electricity data
bucket_name = "nordic-energy-data-pipeline-yh2026"

# Check if the bucket is accessible
response = s3.head_bucket(Bucket=bucket_name)

print("Successfully connected to S3!")


Successfully connected to S3!


In [14]:

# Find the project folder from the current notebook location
PROJECT_ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "json_data_file").is_dir()
     and (folder / "scripts").is_dir()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Project folder not found.")

# Define the local JSON file
local_file = PROJECT_ROOT / "json_data_file" / "prices.json"

# Define where I want to store the file in S3
object_key = "raw/dayaheadprices/prices.json"

# Upload the JSON file to my S3 bucket
s3.upload_file(
    str(local_file),
    bucket_name,
    object_key
)

print("JSON file uploaded successfully!")


JSON file uploaded successfully!


In [15]:

# Check if the JSON file exists in my S3 bucket
response = s3.head_object(
    Bucket=bucket_name,
    Key=object_key
)

# Display the file size in bytes
print("File found in S3!")
print("File size:", response["ContentLength"], "bytes")


File found in S3!
File size: 260198 bytes


In [17]:
# Define the local files I want to upload to S3
local_file = PROJECT_ROOT / "json_data_file" / "prices.json"

parquet_file = PROJECT_ROOT / "data" / "processed" / "day_ahead_prices.parquet"

In [18]:

# Get the current date in Danish time
today = datetime.now(ZoneInfo("Europe/Copenhagen")).date()

# Define the S3 paths using the date
raw_key = f"raw/dayaheadprices/ingestion_date={today}/prices.json"
processed_key = f"processed/dayaheadprices/ingestion_date={today}/day_ahead_prices.parquet"

# Upload the raw JSON file
s3.upload_file(
    str(local_file),
    bucket_name,
    raw_key
)

# Upload the processed Parquet file
s3.upload_file(
    str(parquet_file),
    bucket_name,
    processed_key
)

print("Files uploaded successfully!")
print("Raw:", raw_key)
print("Processed:", processed_key)


Files uploaded successfully!
Raw: raw/dayaheadprices/ingestion_date=2026-11-13/prices.json
Processed: processed/dayaheadprices/ingestion_date=2026-11-13/day_ahead_prices.parquet
